# LAB 6 — Análise de Trace: Identificar Gargalos e Propor Otimizações
## MBA RAG & CAG Aplicados a Direito e Segurança Pública — Aula 3

**Objetivo:** Analisar os traces gerados no LAB 5, identificar o módulo mais lento
e propor otimizações com base nos dados de latência.

**Entregáveis:**
- Tabela de latência por módulo com identificação do gargalo
- Proposta de otimização fundamentada
- Implementação de UMA otimização com medição de impacto

**Tempo estimado:** 20 minutos


## ⚙️ Passo 1 — Setup

In [1]:
import os, json, time
import pandas as pd
from typing import Dict, List

# Para análise real: conectar ao LangFuse e puxar traces
# Para demonstração: usar dados simulados realistas
print("✅ Setup concluído")
print()
print("🎯 Objetivo deste lab: analisar latência e identificar gargalos")

✅ Setup concluído

🎯 Objetivo deste lab: analisar latência e identificar gargalos


## 📊 Passo 2 — Carregar Dados de Latência

In [2]:
# Dados de latência simulados baseados em execuções reais típicas
# Em produção: busque estes dados via LangFuse API
# from langfuse import Langfuse
# lf = Langfuse(...)
# traces = lf.fetch_traces(tags=["advanced_rag", "aula3"])

dados_latencia = [
    {
        "query": "O suspeito pode ficar calado no interrogatório?",
        "spans": {
            "query_rewriting": 1850,   # ms
            "retrieval": 312,
            "reranking": 4230,          # ← GARGALO típico
            "geracao_resposta": 3100
        }
    },
    {
        "query": "Qual é o prazo para audiência de custódia?",
        "spans": {
            "query_rewriting": 1720,
            "retrieval": 287,
            "reranking": 4890,          # ← GARGALO
            "geracao_resposta": 2890
        }
    },
    {
        "query": "O policial precisa de mandado para entrar na casa?",
        "spans": {
            "query_rewriting": 1980,
            "retrieval": 334,
            "reranking": 3950,          # ← GARGALO
            "geracao_resposta": 3420
        }
    }
]

# Calcular totais
for d in dados_latencia:
    d["total"] = sum(d["spans"].values())
    d["pct_reranking"] = round(d["spans"]["reranking"] / d["total"] * 100, 1)

print("✅ Dados carregados:")
for d in dados_latencia:
    print(f"   '{d['query'][:45]}...': total {d['total']}ms (reranking: {d['pct_reranking']}%)")

✅ Dados carregados:
   'O suspeito pode ficar calado no interrogatóri...': total 9492ms (reranking: 44.6%)
   'Qual é o prazo para audiência de custódia?...': total 9787ms (reranking: 50.0%)
   'O policial precisa de mandado para entrar na ...': total 9684ms (reranking: 40.8%)


## 📈 Passo 3 — Análise por Módulo

In [3]:
print("📊 LATÊNCIA MÉDIA POR MÓDULO")
print("=" * 55)
print()

modulos = ["query_rewriting", "retrieval", "reranking", "geracao_resposta"]
medias = {}
for m in modulos:
    vals = [d["spans"][m] for d in dados_latencia]
    medias[m] = sum(vals) / len(vals)

total_medio = sum(medias.values())

# Barra de progresso visual
for m, ms in sorted(medias.items(), key=lambda x: x[1], reverse=True):
    pct = ms / total_medio * 100
    bar = "█" * int(pct / 3) + "░" * (33 - int(pct / 3))
    gargalo = " ← 🔴 GARGALO" if ms == max(medias.values()) else ""
    print(f"  {m:25s}: {ms:6.0f}ms  {pct:5.1f}%  [{bar}]{gargalo}")

print()
print(f"  {'TOTAL':25s}: {total_medio:6.0f}ms  100.0%")
print()

gargalo_modulo = max(medias, key=medias.get)
print(f"🔴 Gargalo identificado: '{gargalo_modulo}'")
print(f"   Latência média: {medias[gargalo_modulo]:.0f}ms")
print(f"   % do tempo total: {medias[gargalo_modulo]/total_medio*100:.1f}%")

📊 LATÊNCIA MÉDIA POR MÓDULO

  reranking                :   4357ms   45.1%  [███████████████░░░░░░░░░░░░░░░░░░] ← 🔴 GARGALO
  geracao_resposta         :   3137ms   32.5%  [██████████░░░░░░░░░░░░░░░░░░░░░░░]
  query_rewriting          :   1850ms   19.2%  [██████░░░░░░░░░░░░░░░░░░░░░░░░░░░]
  retrieval                :    311ms    3.2%  [█░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]

  TOTAL                    :   9654ms  100.0%

🔴 Gargalo identificado: 'reranking'
   Latência média: 4357ms
   % do tempo total: 45.1%


## 💡 Passo 4 — Proposta de Otimizações

In [4]:
print("💡 ESTRATÉGIAS DE OTIMIZAÇÃO — Módulo Reranking")
print("=" * 60)
print()

otimizacoes = [
    {
        "estrategia": "1. Reduzir top-K de entrada",
        "descricao": "Passar 10 docs ao reranker em vez de 20",
        "implementacao": "reranking_instrumentado(query, docs, top_k_input=10)",
        "impacto_esperado": "-50% latência de reranking",
        "risco": "Possível perda de recall se docs relevantes estiverem além do top-10"
    },
    {
        "estrategia": "2. Quantização do modelo (INT8)",
        "descricao": "Carregar BGE-Reranker em INT8 em vez de FP32",
        "implementacao": "model = AutoModel.from_pretrained(model, load_in_8bit=True)",
        "impacto_esperado": "-30% latência, -75% uso de VRAM",
        "risco": "Leve redução de precisão (~1-2% no score)"
    },
    {
        "estrategia": "3. Cache de pares query-doc frequentes",
        "descricao": "Cachear scores de pares que se repetem",
        "implementacao": "from functools import lru_cache → cache(query, doc_id)",
        "impacto_esperado": "-60% latência para queries repetidas",
        "risco": "Cache invalidation ao atualizar corpus"
    },
    {
        "estrategia": "4. Batch assíncrono com limite de tamanho",
        "descricao": "Processar docs em batches de 4 com asyncio",
        "implementacao": "asyncio.gather(*[rerank_batch(b) for b in batches])",
        "impacto_esperado": "-20% latência com paralelismo em GPU",
        "risco": "Complexidade de implementação"
    }
]

for o in otimizacoes:
    print(f"  {o['estrategia']}")
    print(f"    Descrição:  {o['descricao']}")
    print(f"    Código:     {o['implementacao']}")
    print(f"    Esperado:   {o['impacto_esperado']}")
    print(f"    Risco:      {o['risco']}")
    print()

💡 ESTRATÉGIAS DE OTIMIZAÇÃO — Módulo Reranking

  1. Reduzir top-K de entrada
    Descrição:  Passar 10 docs ao reranker em vez de 20
    Código:     reranking_instrumentado(query, docs, top_k_input=10)
    Esperado:   -50% latência de reranking
    Risco:      Possível perda de recall se docs relevantes estiverem além do top-10

  2. Quantização do modelo (INT8)
    Descrição:  Carregar BGE-Reranker em INT8 em vez de FP32
    Código:     model = AutoModel.from_pretrained(model, load_in_8bit=True)
    Esperado:   -30% latência, -75% uso de VRAM
    Risco:      Leve redução de precisão (~1-2% no score)

  3. Cache de pares query-doc frequentes
    Descrição:  Cachear scores de pares que se repetem
    Código:     from functools import lru_cache → cache(query, doc_id)
    Esperado:   -60% latência para queries repetidas
    Risco:      Cache invalidation ao atualizar corpus

  4. Batch assíncrono com limite de tamanho
    Descrição:  Processar docs em batches de 4 com asyncio
    Código:

## 🔧 Passo 5 — Implementar Otimização 1: Reduzir Top-K

In [5]:
# Implementar e medir a Estratégia 1: reduzir top-K de entrada

import time
import random

def simular_reranking(num_docs: int, latencia_base_ms: float = 200) -> tuple:
    """
    Simula o reranking com latência proporcional ao número de documentos.
    
    Latência real ≈ O(N) onde N = número de pares (query, doc) processados.
    Reduzir N de 20 para 10 reduz ~50% a latência.
    """
    # Simular latência proporcional ao número de docs
    latencia = latencia_base_ms * num_docs / 10
    time.sleep(latencia / 1000)  # converter ms para segundos
    
    # Simular resultado: top_5 dos docs
    docs_out = [{"id": f"doc_{i}", "score_reranker": round(random.uniform(0.3, 0.97), 3)} 
                for i in range(min(5, num_docs))]
    return docs_out, latencia

print("🧪 EXPERIMENTO: Impacto da Redução de Top-K no Reranking")
print("=" * 60)
print()

configs = [
    {"top_k_input": 20, "descricao": "Configuração atual (Aula 2 baseline)"},
    {"top_k_input": 10, "descricao": "Otimização 1: top-10"},
    {"top_k_input": 5,  "descricao": "Otimização agressiva: top-5"},
]

resultados_experimento = []

for config in configs:
    k = config["top_k_input"]
    latencias_iter = []
    
    for _ in range(3):  # 3 iterações
        _, lat = simular_reranking(k, latencia_base_ms=200)
        latencias_iter.append(lat)
    
    lat_media = sum(latencias_iter) / len(latencias_iter)
    resultados_experimento.append({
        "config": config["descricao"],
        "top_k": k,
        "latencia_media_ms": round(lat_media, 1)
    })
    
    print(f"  top_k={k:2d} | {lat_media:.0f}ms | {config['descricao']}")

print()
lat_base = resultados_experimento[0]["latencia_media_ms"]
for r in resultados_experimento[1:]:
    reducao = (lat_base - r["latencia_media_ms"]) / lat_base * 100
    print(f"  ✅ top_k={r['top_k']}: redução de {reducao:.0f}% vs baseline")
print()
print("⚠️  TRADE-OFF: Reduzir top-K pode impactar recall!")
print("   Monitore: algum documento relevante foi excluído com top-K=10?")

🧪 EXPERIMENTO: Impacto da Redução de Top-K no Reranking

  top_k=20 | 400ms | Configuração atual (Aula 2 baseline)
  top_k=10 | 200ms | Otimização 1: top-10
  top_k= 5 | 100ms | Otimização agressiva: top-5

  ✅ top_k=10: redução de 50% vs baseline
  ✅ top_k=5: redução de 75% vs baseline

⚠️  TRADE-OFF: Reduzir top-K pode impactar recall!
   Monitore: algum documento relevante foi excluído com top-K=10?


## 📝 Passo 6 — Recomendação Final

In [6]:
print("📋 RELATÓRIO DE ANÁLISE DE PERFORMANCE")
print("=" * 60)
print()
print("PIPELINE: Advanced RAG Jurídico — Aula 3")
print()
print("GARGALO IDENTIFICADO:")
print("  Módulo: reranking (BGE-Reranker-v2-m3)")
print("  Latência média: 4.356ms (~43% do tempo total)")
print("  Causa raiz: Cross-encoder processa N pares (query, doc) sequencialmente")
print()
print("RECOMENDAÇÕES (por prioridade):")
print()
print("  1ª PRIORIDADE — Reduzir top-K (Baixo risco, alto impacto):")
print("     Reduzir de top-20 para top-10 antes do reranking")
print("     Ganho esperado: -50% latência de reranking (~2s)")
print("     Implementação: 1 linha de código")
print()
print("  2ª PRIORIDADE — Quantização INT8 (Médio risco, alto impacto):")
print("     Carregar modelo em INT8 com bitsandbytes")
print("     Ganho esperado: -30% latência + -75% VRAM")
print("     Ideal para deploys em produção com múltiplas requisições")
print()
print("  3ª PRIORIDADE — Cache de resultados (Baixo risco para queries repetidas):")
print("     Implementar cache com TTL de 1h para scores de pares frequentes")
print("     Ganho: -60% latência para queries repetidas (típico em ambientes policiias)")
print()
print("LATÊNCIA TOTAL ATUAL vs OTIMIZADA (estimativa):")
print("  Atual:      ~10.300ms (query_rew: 1.850 | ret: 311 | rerank: 4.357 | gen: 3.137)")
print("  Otimizado:  ~7.600ms  (redução ~26% com estratégia 1)")
print()
print("✅ LAB 6 CONCLUÍDO!")
print()
print("📋 CHECKLIST DE ENTREGA:")
print("   [ ] Gargalo identificado com dados quantitativos")
print("   [ ] Tabela de latência por módulo (% do total)")
print("   [ ] 3 estratégias de otimização propostas")
print("   [ ] Experimento de top-K executado com medição de impacto")
print("   [ ] Recomendação final documentada com justificativa")

📋 RELATÓRIO DE ANÁLISE DE PERFORMANCE

PIPELINE: Advanced RAG Jurídico — Aula 3

GARGALO IDENTIFICADO:
  Módulo: reranking (BGE-Reranker-v2-m3)
  Latência média: 4.356ms (~43% do tempo total)
  Causa raiz: Cross-encoder processa N pares (query, doc) sequencialmente

RECOMENDAÇÕES (por prioridade):

  1ª PRIORIDADE — Reduzir top-K (Baixo risco, alto impacto):
     Reduzir de top-20 para top-10 antes do reranking
     Ganho esperado: -50% latência de reranking (~2s)
     Implementação: 1 linha de código

  2ª PRIORIDADE — Quantização INT8 (Médio risco, alto impacto):
     Carregar modelo em INT8 com bitsandbytes
     Ganho esperado: -30% latência + -75% VRAM
     Ideal para deploys em produção com múltiplas requisições

  3ª PRIORIDADE — Cache de resultados (Baixo risco para queries repetidas):
     Implementar cache com TTL de 1h para scores de pares frequentes
     Ganho: -60% latência para queries repetidas (típico em ambientes policiias)

LATÊNCIA TOTAL ATUAL vs OTIMIZADA (estimativa

## 📝 Análise Final — Complete aqui

**Módulo identificado como gargalo:**  
→ *[resposta]*

**Estratégia de otimização escolhida e justificativa:**  
→ *[resposta]*

**Impacto medido no experimento (ms e %):**  
→ *[resposta]*

**Existe trade-off de qualidade? Como mitigá-lo?**  
→ *[resposta]*
